# 🧪 Praktikum 06 – Advanced Web-Scraping & Embedding-Engineering
**Applied Generative AI – NLP | Sommersemester 2026**

> 🎯 **Lernziele:** Rekursive Scraping-Strategien · Token-basierte Chunking-Verfahren · Vektorraum-Analyse und Distanzmetriken

⏱️ **Dauer:** 90 Minuten

In [ ]:
import importlib
import os
import shutil
import subprocess
import sys
import time
from importlib.util import find_spec

IN_COLAB = "google.colab" in sys.modules
MODEL = os.getenv("LLM_MODEL", "qwen3.5:0.8b").strip()
EMBED_MODEL = os.getenv("EMBED_MODEL", "nomic-embed-text").strip()
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://127.0.0.1:11434").strip()

REQUIRED = {
    "requests": ("requests", "2.33.1"),
    "bs4": ("beautifulsoup4", "4.14.3"),
    "ollama": ("ollama", "0.6.1"),
    "numpy": ("numpy", "2.4.4"),
    "sklearn": ("scikit-learn", "1.8.0"),
    "matplotlib": ("matplotlib", "3.10.8"),
    "pandas": ("pandas", "3.0.2"),
    "tiktoken": ("tiktoken", "0.12.0"),
}


def run_install(specs):
    if shutil.which("uv") is None:
        raise RuntimeError("uv is not installed. Install uv and rerun the setup cell.")

    command = ["uv", "pip", "install", "--python", sys.executable, *specs]
    in_venv = sys.prefix != getattr(sys, "base_prefix", sys.prefix) or bool(os.environ.get("VIRTUAL_ENV"))
    if not in_venv:
        command.append("--system")

    print("$", " ".join(command))
    subprocess.check_call(command)


missing_specs = [
    f"{install_name}=={version}"
    for import_name, (install_name, version) in REQUIRED.items()
    if find_spec(import_name) is None
]
if missing_specs:
    run_install(missing_specs)
    importlib.invalidate_caches()

missing_after = [import_name for import_name in REQUIRED if find_spec(import_name) is None]
if missing_after:
    raise RuntimeError(f"Missing packages after installation: {', '.join(missing_after)}")

import requests
from bs4 import BeautifulSoup
import matplotlib.pyplot as plt
import numpy as np
import ollama
import tiktoken


def is_local_host(url):
    host = url.split("://", 1)[-1].split("/", 1)[0].split(":", 1)[0].lower()
    return host in {"127.0.0.1", "localhost", "0.0.0.0"}


def wait_for_ollama(base_url, timeout=90):
    deadline = time.time() + timeout
    last_error = None
    while time.time() < deadline:
        try:
            response = requests.get(f"{base_url.rstrip('/')}/api/tags", timeout=5)
            response.raise_for_status()
            return response.json()
        except Exception as exc:
            last_error = exc
            time.sleep(2)
    raise RuntimeError(f"Ollama server at {base_url} is not reachable: {last_error}")


def model_is_available(requested_name, available_names):
    candidates = {requested_name, f"{requested_name}:latest"}
    return any(candidate in available_names for candidate in candidates)


if not is_local_host(OLLAMA_BASE_URL):
    raise RuntimeError("This notebook expects a local Ollama service via OLLAMA_BASE_URL.")

if shutil.which("ollama") is None:
    if not IN_COLAB:
        raise RuntimeError("Ollama is not installed locally. Install Ollama and rerun the setup cell.")
    subprocess.check_call(["bash", "-lc", "curl -fsSL https://ollama.com/install.sh | sh"])

try:
    wait_for_ollama(OLLAMA_BASE_URL, timeout=5)
except RuntimeError:
    ollama_log = open("/tmp/ollama-notebook.log", "a", encoding="utf-8")
    subprocess.Popen(
        ["ollama", "serve"],
        stdout=ollama_log,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    wait_for_ollama(OLLAMA_BASE_URL, timeout=90)

os.environ["OLLAMA_HOST"] = OLLAMA_BASE_URL
env = dict(os.environ)
for model_name in [MODEL, EMBED_MODEL]:
    subprocess.check_call(["ollama", "pull", model_name], env=env)

payload = wait_for_ollama(OLLAMA_BASE_URL, timeout=30)
available_models = [item.get("name", "") for item in payload.get("models", []) if isinstance(item, dict)]
missing_models = [
    model_name
    for model_name in [MODEL, EMBED_MODEL]
    if not model_is_available(model_name, available_models)
]
if missing_models:
    raise RuntimeError(f"Missing local Ollama models: {', '.join(missing_models)}")

print("Runtime:", "Google Colab" if IN_COLAB else "Local/Jupyter")
print("Models:", MODEL, ",", EMBED_MODEL)
print("Available local models:", ", ".join(available_models))

## Teil 1 – Strukturiertes Scraping & Cleaning ⏱️ 25 min
Wir extrahieren Content, Metadaten und bereinigen den Text.

In [ ]:
import re


def normalize_text(text):
    return re.sub(r"\s+", " ", text).strip()


def clean_section_title(text):
    text = normalize_text(text)
    text = re.sub(r"\[\s*Bearbeiten\s*\|\s*Quelltext bearbeiten\s*\]$", "", text)
    return normalize_text(text)


def extract_section_data(section, data):
    heading_container = next(
        (
            child
            for child in section.children
            if getattr(child, "name", None) == "div" and "mw-heading" in (child.get("class") or [])
        ),
        None,
    )

    title = "Einleitung"
    if heading_container is not None:
        heading = heading_container.find(re.compile(r"^h[1-6]$"), recursive=False)
        if heading is not None:
            title = clean_section_title(heading.get_text(" ", strip=True))

    text_parts = []
    for child in section.children:
        if not getattr(child, "name", None):
            continue
        if child == heading_container or child.name == "section":
            continue
        if child.name == "div" and "hauptartikel" in (child.get("class") or []):
            continue
        if child.name in {"p", "ul", "ol", "blockquote", "dl"}:
            block_text = normalize_text(child.get_text(" ", strip=True))
            if block_text:
                text_parts.append(block_text)

    text = " ".join(text_parts)
    if len(text) > 100:
        data.append({"title": title, "content": text})

    for nested_section in section.find_all("section", recursive=False):
        extract_section_data(nested_section, data)


def advanced_scrape(url):
    response = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=10)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")

    content_root = soup.find(class_="mw-parser-output")
    if content_root is None:
        raise RuntimeError("Could not find the main article content on the page.")

    data = []
    for top_section in content_root.find_all("section", recursive=False):
        extract_section_data(top_section, data)

    if not data:
        raise RuntimeError("No suitable sections were extracted from the page.")
    return data


raw_data = advanced_scrape("https://de.wikipedia.org/wiki/K%C3%BCnstliche_Intelligenz")
print(f"Gefundene Sektionen: {len(raw_data)}")

## Teil 2 – Token-basiertes Chunking ⏱️ 30 min
Wir nutzen `tiktoken` (OpenAI Standard).

In [ ]:
def token_chunker(text, max_tokens=100, overlap=20):
    enc = tiktoken.get_encoding("cl100k_base")
    tokens = enc.encode(text)
    chunks = []
    for i in range(0, len(tokens), max_tokens - overlap):
        chunks.append(enc.decode(tokens[i : i + max_tokens]))
    return chunks

all_chunks = []
for sec in raw_data:
    for c in token_chunker(sec['content']): all_chunks.append({"source": sec['title'], "text": c})

print(f"Anzahl Chunks: {len(all_chunks)}")
print(f"Beispiel-Chunk:\n{all_chunks[0]['text'][:100]}...")

## Teil 3 – Vektorraum-Analyse ⏱️ 35 min
Visualisierung semantischer Cluster.

In [ ]:
subset = all_chunks[:20]
vectors = np.array([ollama.embeddings(model=EMBED_MODEL, prompt=c['text'])['embedding'] for c in subset])

from sklearn.metrics.pairwise import cosine_distances
dist_matrix = cosine_distances(vectors)

plt.imshow(dist_matrix, cmap='viridis')
plt.title("Ähnlichkeitsmatrix")
plt.show()

## 🧩 Aufgaben
1. Vergleiche `Fixed Size` vs `Sliding Window` Chunking.
2. Implementiere eine Top-3 Vektorsuche.
3. Warum sind Token-Grenzen besser als Zeichen-Grenzen?